# Volatility MCP — Client Demo

This notebook shows how to connect to the Volatility MCP server as a client,
call tools, and follow the feedback loop programmatically.

**Prerequisites:**
```bash
pip install mcp
python -m mcp_server.server --transport sse --port 8765 &
```

In [ ]:
# Connect to the MCP server (SSE transport)
import asyncio
from mcp import ClientSession, SseServerParameters
from mcp.client.sse import sse_client

async def connect():
    params = SseServerParameters(
        url='http://localhost:8765/sse',
    )
    async with sse_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # List available tools
            tools = await session.list_tools()
            print(f'Available tools: {len(tools.tools)}')
            for t in tools.tools[:5]:
                print(f'  - {t.name}: {t.description[:80]}...')
            print('  ...')
            
            return session

session = await connect()

## 1. Open a session and load data

In [ ]:
# Open a session
result = await session.call_tool('open_session', {})
session_id = result.content[0].text.split('"session_id": "')[1].split('"')[0]
print(f'Session ID: {session_id}')

# Load NIFTY 50 data
result = await session.call_tool('load_market', {
    'session_id': session_id,
    'ticker': '^NSEI',
    'years': 5,
    'frequency': 'Daily',
})
print(result.content[0].text)

## 2. Run pre-flight checks

In [ ]:
result = await session.call_tool('run_preflight', {
    'session_id': session_id,
})
print(result.content[0].text[:500])

## 3. Ask the advisor what to do next

In [ ]:
result = await session.call_tool('get_next_action', {
    'session_id': session_id,
})
print(result.content[0].text)

## 4. Optimize, fit, and compute VaR — following the advisor

In [ ]:
# Optimize order
result = await session.call_tool('optimize_order', {
    'session_id': session_id,
    'model_family': 'GARCH',
    'max_p': 2,
    'max_q': 1,
})
print('Order optimized')

# Fit GARCH
result = await session.call_tool('fit_model', {
    'session_id': session_id,
    'model': 'GARCH',
})
print(result.content[0].text[:300])

# Compute VaR
result = await session.call_tool('compute_var', {
    'session_id': session_id,
    'model': 'GARCH',
    'confidence': 0.975,
    'method': 'student_t',
    'portfolio_value': 1000000,
})
print(result.content[0].text[:300])

## 5. Generate a Markdown report

In [ ]:
result = await session.call_tool('build_markdown', {
    'session_id': session_id,
})
report = result.content[0].text
print(f'Report length: {len(report)} chars')
print(report[:500])

## 6. View the full decision audit trail

In [ ]:
result = await session.call_tool('explain_decision', {
    'session_id': session_id,
})
print(result.content[0].text[:500])